# 04. Token Sequence Corpus and Labels

목표는 Phase 1에서 확정한 `kmeans_boundary_aware K=32` tokenizer를 고정하고, 8개 dataset의 token sequence corpus와 label store를 생성한 뒤 bigram 순차 구조가 surrogate 우연 수준을 넘는지 판정하는 것이다.

Contract:
- tokenizer 재튜닝 금지: continuous K=32 + boundary 8 + zero-range 1 = vocab 41
- canonical corpus seed는 7, 안정성 보고 seeds는 7/17/37
- 모든 fit은 train split의 interior x interior row에서만 수행
- label은 t+1 이후 정보만 사용하고 split boundary/embargo/horizon crossing row는 null 처리
- Williams Fractal은 저장만 하며 Phase 2 gate 분석에는 사용하지 않는다


In [1]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from kairos.experiments.artifacts import config_hash, find_project_root, write_csv, write_json
from kairos.experiments.protocol import split_protocol_for_interval
from kairos.experiments.shape_tokenizer.baselines import (
    FEATURE_INPUTS,
    MERGED_FEATURE_INPUTS,
    component_feature_runs,
    dataset_interval,
    latest_feature_run,
)
from kairos.experiments.shape_tokenizer.corpus import (
    VOCAB_BOUNDARY,
    VOCAB_FULL,
    VOCAB_INTERIOR,
    build_corpus_rows,
    compute_williams_fractals,
    grouped_token_sequences,
    information_gain,
    jensen_shannon_distance,
    join_shape_ohlc,
    ohlc_rows_from_candles,
    read_shape_sample_rows,
    surrogate_information_gain,
    token_distribution,
    token_rows_by_seed,
    unigram_kl,
    assign_coarse_classes,
)
from kairos.experiments.shape_tokenizer.feature_validation import D1_FEATURE_REQUESTS
from kairos.experiments.shape_tokenizer.labels import compute_label_rows
from kairos.sources.brokers import fetch_index_candles

PROJECT_ROOT = find_project_root()
RUNS_ROOT = PROJECT_ROOT / "notebooks" / "runs" / "candlestick-shape-quantization"
PHASE_02_ID = "phase-02-token-corpus"
STEP_ID = "step-01-corpus-and-labels"
SOURCE_NOTEBOOK = "notebooks/candlestick-shape-quantization/04_token_corpus.ipynb"
SEEDS = (7, 17, 37)
CANONICAL_SEED = 7
CODEBOOK_SIZE = 32
VOCAB_SIZE = 41
SURROGATE_REPEATS = 100

_feature_by_id = {item.dataset_id: item for item in FEATURE_INPUTS}
_request_by_dataset_id = {request.dataset_id: request for request in D1_FEATURE_REQUESTS}
D1_TARGETS = (
    _feature_by_id["d1_kospi_daily"],
    _feature_by_id["d1_kosdaq_daily"],
    _feature_by_id["d1_nasdaq_daily"],
    _feature_by_id["d1_spx_daily"],
    _feature_by_id["d1_kospi_1m"],
    _feature_by_id["d1_kosdaq_1m"],
)
D2_TARGETS = MERGED_FEATURE_INPUTS
TARGETS = (*D1_TARGETS, *D2_TARGETS)
DATASET_IDS = tuple(item.dataset_id for item in TARGETS)
DATASET_IDS


('d1_kospi_daily',
 'd1_kosdaq_daily',
 'd1_nasdaq_daily',
 'd1_spx_daily',
 'd1_kospi_1m',
 'd1_kosdaq_1m',
 'd2_kr-kospi-kosdaq_daily',
 'd2_kr-kospi-kosdaq_1m')

In [2]:
def dataset_components(dataset: Any) -> tuple[Any, ...]:
    return tuple(dataset.components) if hasattr(dataset, "components") else (dataset,)


def load_shape_rows(dataset: Any) -> list[dict[str, Any]]:
    if hasattr(dataset, "components"):
        rows: list[dict[str, Any]] = []
        run_dirs = component_feature_runs(RUNS_ROOT, dataset)
        for component in dataset.components:
            component_rows = read_shape_sample_rows(run_dirs[component.dataset_id])
            rows.extend(row | {"source_dataset_id": component.dataset_id} for row in component_rows)
        return sorted(rows, key=lambda row: (row["symbol"], row["timestamp"]))
    run_dir = latest_feature_run(RUNS_ROOT, dataset)
    return sorted(read_shape_sample_rows(run_dir), key=lambda row: (row["symbol"], row["timestamp"]))


async def fetch_ohlc_rows(dataset: Any) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for component in dataset_components(dataset):
        request = _request_by_dataset_id[component.dataset_id]
        candles = await fetch_index_candles(request, env_path=PROJECT_ROOT / ".env")
        rows.extend(ohlc_rows_from_candles(candles))
    return sorted(rows, key=lambda row: (row["symbol"], row["timestamp"]))


def phase2_config(dataset: Any) -> dict[str, Any]:
    interval = dataset_interval(dataset.dataset_id)
    split = split_protocol_for_interval(interval)
    return {
        "source_notebook": SOURCE_NOTEBOOK,
        "phase": PHASE_02_ID,
        "step": STEP_ID,
        "dataset_id": dataset.dataset_id,
        "components": [component.dataset_id for component in dataset_components(dataset)],
        "tokenizer": {
            "model": "kmeans_boundary_aware",
            "continuous_codebook_size": CODEBOOK_SIZE,
            "boundary_discrete_tokens": 8,
            "zero_range_special_token": 1,
            "vocabulary_size": VOCAB_SIZE,
            "fit_rows": "train split and interior x interior only",
            "canonical_seed": CANONICAL_SEED,
            "stability_seeds": list(SEEDS),
        },
        "labels": {
            "horizons": list(split.label_horizons),
            "direction_threshold": "dataset/horizon train split median abs forward log return",
            "rv": "Parkinson sqrt(mean(log(high/low)^2 / (4 log 2)))",
            "vol_expansion": "fwd_rv_h / trailing_rv_h > 1.5",
            "drawdown_event_20": "train split max drawdown 90% quantile",
            "embargo_days": split.embargo_days,
            "regime": "daily only: close vs 200-bar MA x train trailing vol tercile",
        },
        "fractal": {
            "name": "Williams Fractal",
            "n": 2,
            "tie_rule": "strict inequality; ties are not pivots",
            "minute_session_rule": "1m pivot windows must stay inside the same trade date",
            "confirmed_at": "pivot timestamp plus the n=2-th future candle timestamp",
        },
        "sequence_gate": {
            "surrogate_repeats": SURROGATE_REPEATS,
            "vocab_decomposition": [VOCAB_FULL, VOCAB_INTERIOR, VOCAB_BOUNDARY],
            "train_split_is_gate": True,
        },
        "split": split,
    }


In [3]:
def token_seed_stability(tokens_by_seed: dict[int, np.ndarray]) -> list[dict[str, Any]]:
    canonical = token_distribution(tokens_by_seed[CANONICAL_SEED], vocab_size=VOCAB_SIZE)
    rows = []
    for seed, tokens in tokens_by_seed.items():
        distribution = token_distribution(tokens, vocab_size=VOCAB_SIZE)
        rows.append(
            {
                "seed": seed,
                "token_entropy": float(-np.sum(distribution[distribution > 0] * np.log2(distribution[distribution > 0]))),
                "effective_vocab_size": float(np.exp(-np.sum(distribution[distribution > 0] * np.log(distribution[distribution > 0])))),
                "unigram_kl_to_seed7": 0.0 if seed == CANONICAL_SEED else unigram_kl(distribution, canonical),
            }
        )
    return rows


def jsd_rows(corpus_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    groups: dict[tuple[str, str, str], list[int]] = defaultdict(list)
    for row in corpus_rows:
        groups[(row["symbol"], row["split"], str(row.get("regime")))] .append(int(row["token"]))
    rows = []
    keys = sorted(groups)
    distributions = {key: token_distribution(groups[key], vocab_size=VOCAB_SIZE) for key in keys}
    for left in keys:
        for right in keys:
            if left >= right:
                continue
            rows.append(
                {
                    "left": "|".join(left),
                    "right": "|".join(right),
                    "js_distance": jensen_shannon_distance(distributions[left], distributions[right]),
                }
            )
    return rows


def entropy_gate_rows(corpus_rows: list[dict[str, Any]], *, interval: str) -> list[dict[str, Any]]:
    rows = []
    for split_name in ("train", "validation", "test"):
        sequences = grouped_token_sequences(corpus_rows, split=split_name, interval=interval)
        for vocab in (VOCAB_FULL, VOCAB_INTERIOR, VOCAB_BOUNDARY):
            stats = information_gain(sequences, vocab=vocab)
            surrogate = surrogate_information_gain(
                sequences,
                vocab=vocab,
                repeats=SURROGATE_REPEATS,
                seed=CANONICAL_SEED,
            )
            rows.append(
                {
                    "split": split_name,
                    "vocab": vocab,
                    "unigram_entropy": stats.unigram_entropy,
                    "conditional_entropy": stats.conditional_entropy,
                    "information_gain": stats.information_gain,
                    "token_count": stats.token_count,
                    "bigram_count": stats.bigram_count,
                    "surrogate_mean": surrogate["surrogate_mean"],
                    "surrogate_std": surrogate["surrogate_std"],
                    "surrogate_quantile": surrogate["surrogate_quantile"],
                    "z_score": surrogate["z_score"],
                }
            )
    return rows


In [4]:
def plot_token_frequency(corpus_rows: list[dict[str, Any]], figure_dir: Path) -> str:
    figure_dir.mkdir(parents=True, exist_ok=True)
    grouped: dict[tuple[str, str], list[int]] = defaultdict(list)
    for row in corpus_rows:
        grouped[(row["symbol"], row["split"])].append(int(row["token"]))
    labels = sorted(grouped)
    matrix = np.vstack([token_distribution(grouped[label], vocab_size=VOCAB_SIZE) for label in labels])
    fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(labels))))
    image = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
    ax.set_title("Token frequency by symbol and split")
    ax.set_xlabel("token id")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(["|".join(label) for label in labels])
    fig.colorbar(image, ax=ax, label="share")
    fig.tight_layout()
    path = figure_dir / "token_frequency_by_symbol_split.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return str(path)


def plot_jsd_heatmap(jsd: list[dict[str, Any]], figure_dir: Path) -> str:
    labels = sorted({row["left"] for row in jsd} | {row["right"] for row in jsd})
    index = {label: i for i, label in enumerate(labels)}
    matrix = np.zeros((len(labels), len(labels)))
    for row in jsd:
        i = index[row["left"]]
        j = index[row["right"]]
        matrix[i, j] = matrix[j, i] = row["js_distance"]
    fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(labels)), max(5, 0.45 * len(labels))))
    image = ax.imshow(matrix, interpolation="nearest", cmap="magma")
    ax.set_title("Jensen-Shannon distance")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax, label="distance")
    fig.tight_layout()
    path = figure_dir / "jsd_heatmap.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return str(path)


def plot_transition_matrix(corpus_rows: list[dict[str, Any]], figure_dir: Path, *, interval: str) -> str:
    matrix = np.zeros((CODEBOOK_SIZE, CODEBOOK_SIZE), dtype=float)
    for sequence in grouped_token_sequences(corpus_rows, split="train", interval=interval):
        for prev, nxt in zip(sequence, sequence[1:], strict=False):
            if 0 <= prev < CODEBOOK_SIZE and 0 <= nxt < CODEBOOK_SIZE:
                matrix[prev, nxt] += 1
    row_sums = matrix.sum(axis=1, keepdims=True)
    matrix = np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums > 0)
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(matrix, interpolation="nearest", cmap="viridis")
    ax.set_title("Interior-only transition matrix (train, seed 7)")
    ax.set_xlabel("next token")
    ax.set_ylabel("previous token")
    fig.colorbar(image, ax=ax, label="P(next | previous)")
    fig.tight_layout()
    path = figure_dir / "transition_matrix_interior_train.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return str(path)


def plot_information_gain(entropy_rows_: list[dict[str, Any]], figure_dir: Path) -> str:
    rows = [row for row in entropy_rows_ if row["split"] == "train"]
    labels = [row["vocab"] for row in rows]
    observed = [row["information_gain"] for row in rows]
    surrogate = [row["surrogate_mean"] for row in rows]
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, observed, width, label="observed")
    ax.bar(x + width / 2, surrogate, width, label="surrogate mean")
    ax.set_title("Information gain vs surrogate (train)")
    ax.set_ylabel("bits")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    path = figure_dir / "information_gain_vs_surrogate_train.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return str(path)


def make_figures(corpus_rows: list[dict[str, Any]], metrics: dict[str, Any], run_dir: Path, *, interval: str) -> list[str]:
    figure_dir = run_dir / "figures"
    return [
        plot_token_frequency(corpus_rows, figure_dir),
        plot_jsd_heatmap(metrics["jsd_rows"], figure_dir),
        plot_transition_matrix(corpus_rows, figure_dir, interval=interval),
        plot_information_gain(metrics["entropy_gate_rows"], figure_dir),
    ]


In [5]:
async def build_dataset_corpus(dataset: Any) -> dict[str, Any]:
    dataset_id = dataset.dataset_id
    interval = dataset_interval(dataset_id)
    split = split_protocol_for_interval(interval)
    config = phase2_config(dataset)
    cfg_hash = config_hash(config)
    started_at = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    run_dir = RUNS_ROOT / PHASE_02_ID / STEP_ID / dataset_id / f"cfg-{cfg_hash}" / f"run-{started_at}_seed-{CANONICAL_SEED}"
    run_dir.mkdir(parents=True, exist_ok=True)

    shape_rows = load_shape_rows(dataset)
    ohlc_rows = await fetch_ohlc_rows(dataset)
    joined_rows, coverage = join_shape_ohlc(shape_rows, ohlc_rows)
    tokens_by_seed = token_rows_by_seed(joined_rows, seeds=SEEDS, split=split, codebook_size=CODEBOOK_SIZE)
    coarse_classes, coarse_config = assign_coarse_classes(joined_rows, split=split)
    label_rows, label_metrics = compute_label_rows(joined_rows, split=split, interval=interval)
    fractals = compute_williams_fractals(joined_rows, interval=interval, n=2)
    corpus_rows = build_corpus_rows(
        joined_rows,
        label_rows,
        canonical_tokens=tokens_by_seed[CANONICAL_SEED],
        coarse_classes=coarse_classes,
        fractals=fractals,
        split=split,
    )

    fractal_count = sum(1 for row in corpus_rows if row["fractal_high"] or row["fractal_low"])
    entropy_rows_ = entropy_gate_rows(corpus_rows, interval=interval)
    metrics = {
        "dataset_id": dataset_id,
        "interval": interval,
        "cfg_hash": cfg_hash,
        "run_dir": str(run_dir),
        "coverage": coverage,
        "row_count": len(corpus_rows),
        "token_seed_stability": token_seed_stability(tokens_by_seed),
        "coarse_class_config": coarse_config,
        "label_metrics": label_metrics,
        "fractal": {
            "pivot_count": fractal_count,
            "pivot_ratio": fractal_count / len(corpus_rows) if corpus_rows else None,
            "high_count": sum(1 for row in corpus_rows if row["fractal_high"]),
            "low_count": sum(1 for row in corpus_rows if row["fractal_low"]),
        },
        "jsd_rows": jsd_rows(corpus_rows),
        "entropy_gate_rows": entropy_rows_,
    }
    figure_files = make_figures(corpus_rows, metrics, run_dir, interval=interval)
    metrics["figure_files"] = figure_files

    write_json(run_dir / "experiment_config.json", config | {"cfg_hash": cfg_hash})
    write_csv(run_dir / "tables" / "corpus.csv", corpus_rows)
    write_csv(run_dir / "tables" / "token_seed_stability.csv", metrics["token_seed_stability"])
    write_csv(run_dir / "tables" / "entropy_gate.csv", entropy_rows_)
    write_csv(run_dir / "tables" / "jsd.csv", metrics["jsd_rows"])
    write_json(run_dir / "metrics.json", metrics)
    return metrics


async def run_phase2_corpus() -> list[dict[str, Any]]:
    results = []
    for dataset in TARGETS:
        results.append(await build_dataset_corpus(dataset))
    return results


In [6]:
RUN_RESULTS = await run_phase2_corpus()
[(result["dataset_id"], result["row_count"], result["coverage"]["join_coverage"]) for result in RUN_RESULTS]


[('d1_kospi_daily', 9420, 1.0),
 ('d1_kosdaq_daily', 7365, 1.0),
 ('d1_nasdaq_daily', 6664, 1.0),
 ('d1_spx_daily', 5302, 1.0),
 ('d1_kospi_1m', 94149, 1.0),
 ('d1_kosdaq_1m', 94236, 1.0),
 ('d2_kr-kospi-kosdaq_daily', 16785, 1.0),
 ('d2_kr-kospi-kosdaq_1m', 188385, 1.0)]

In [7]:
# Acceptance checks
assert len(RUN_RESULTS) == 8
for result in RUN_RESULTS:
    run_dir = Path(result["run_dir"])
    assert result["coverage"]["join_coverage"] >= 0.995, result["dataset_id"]
    assert (run_dir / "tables" / "corpus.csv").exists()
    assert (run_dir / "metrics.json").exists()
    assert len(result["figure_files"]) == 4
    assert all(Path(path).exists() for path in result["figure_files"])
    assert {row["vocab"] for row in result["entropy_gate_rows"]} == {VOCAB_FULL, VOCAB_INTERIOR, VOCAB_BOUNDARY}
    train_entropy = [row for row in result["entropy_gate_rows"] if row["split"] == "train"]
    assert len(train_entropy) == 3
    corpus_path = run_dir / "tables" / "corpus.csv"
    import csv
    with corpus_path.open(newline="", encoding="utf-8") as handle:
        corpus = list(csv.DictReader(handle))
    assert corpus
    for row in corpus:
        pivot = row["fractal_high"] == "True" or row["fractal_low"] == "True"
        assert (row["fractal_confirmed_at"] != "") == pivot
        if pivot:
            assert row["fractal_confirmed_at"] > row["timestamp"]
        for horizon in (1, 5, 20):
            if row.get(f"label_embargoed_{horizon}") == "True":
                assert row.get(f"fwd_log_return_{horizon}") in {"", "None"}
